# Network-Based Policy Candidates Selection

Final project — CIVIL 534. Translates the World3 system dynamics model into a directed graph and analyzes structure: centrality, cycles, communities (with k-core / k-component decomposition), and percolation robustness. Findings feed back into policy development.

In [46]:
import inspect
import json
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd


## 1. World3 Network Metrics Calculation


- the directed World3 graph `G`
- centrality metrics (`in_degree`, `out_degree`, `betweenness`, `pagerank`, `eigenvector`)
- community assignments from greedy modularity on the undirected projection
- `k`-core values from the undirected projection
- directed cycle participation, used later for `feedbackimportance`



In [47]:
with open('../world3-03_variables.json') as f:
    w3_vars = json.load(f)

G = nx.DiGraph()
for name, val in w3_vars.items():
    G.add_node(name, var_type=val['type'])
    deps = val.get('dependencies') or []
    G.add_edges_from((dep, name) for dep in deps)

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
print(f'Is DAG: {nx.is_directed_acyclic_graph(G)}')


Nodes: 316, Edges: 507
Is DAG: False


### Structural metrics

These metrics are reused directly by the target-specific upstream screening:

- `betweenness`: bridge importance in the directed graph
- `pagerank`: directed structural importance
- `eigenvector`: embeddedness in the undirected backbone
- `community`: subsystem annotation from modularity detection
- `core`: density / backbone membership from the undirected `k`-core decomposition


In [48]:
from networkx.algorithms.community import greedy_modularity_communities

U = G.to_undirected()

centrality = pd.DataFrame({
    'var_type': pd.Series(nx.get_node_attributes(G, 'var_type')),
    'in_degree': pd.Series(dict(G.in_degree())),
    'out_degree': pd.Series(dict(G.out_degree())),
    'betweenness': pd.Series(nx.betweenness_centrality(G)),
    'pagerank': pd.Series(nx.pagerank(G)),
    'eigenvector': pd.Series(nx.eigenvector_centrality(U, max_iter=1000)),
})

communities = list(greedy_modularity_communities(U))
node_to_comm = {n: i for i, comm in enumerate(communities) for n in comm}

U_simple = U.copy()
U_simple.remove_edges_from(nx.selfloop_edges(U_simple))
core_series = pd.Series(nx.core_number(U_simple), name='core').sort_values(ascending=False)

print(f'Communities: {len(communities)}')
print(f'Highest k-core: {core_series.max()}')


Communities: 17
Highest k-core: 3


### Directed feedback cycles 

The leverage score later includes a feedback term. To support that term, we precompute the directed simple cycles and count how many cycles each node participates in.


In [49]:
cycles = list(nx.simple_cycles(G))
cycle_membership = {}
for cycle in cycles:
    for node in cycle:
        cycle_membership[node] = cycle_membership.get(node, 0) + 1

cycle_membership = pd.Series(cycle_membership, name='cycles').sort_values(ascending=False)
print(f'Total simple cycles: {len(cycles)}')
print('Top cycle-membership nodes:')
display(cycle_membership.head(10))


Total simple cycles: 1863803
Top cycle-membership nodes:


population                            1831276
life_expectancy                       1821608
industrial_output                     1791059
arable_land                           1584918
land_yield                            1534184
population_0_to_14                    1520295
agricultural_input_per_hectare        1491397
labor_utilization_fraction            1490236
delayed_labor_utilization_fraction    1490236
capacity_utilization_fraction         1490236
Name: cycles, dtype: int64

## 2. Target-specific upstream candidate selection

This section selects candidate influence factors for a later grid search without assuming the previous four policy levers. The targets are the four sustainability outcomes used in the project:

- `nr` → `nonrenewable_resources`
- `pop` → `population`
- `ppolx` → `persistent_pollution_index`
- `le` → `life_expectancy`

The workflow is deliberately target-specific: first locate each target in the network, then find directed ancestors that can reach those targets, then filter those ancestors to variables that can plausibly be used as policy/grid-search factors. Community is used as annotation, not as a hard filter, because important upstream variables can affect a target across community boundaries.


In [50]:
# Target definitions and target-level network metrics
# Load full-name/abbreviation metadata from the World3 JSON file.
metadata_paths = [
    Path('../world3-03_variables.json'),
    Path('../../world3-03_variables.json'),
]
for metadata_path in metadata_paths:
    if metadata_path.exists():
        with open(metadata_path) as f:
            variable_meta = json.load(f)
        break
else:
    raise FileNotFoundError('world3-03_variables.json not found for target-specific analysis')

targets = {
    'nr':    'nonrenewable_resources',
    'pop':   'population',
    'ppolx': 'persistent_pollution_index',
    'le':    'life_expectancy',
}

target_nodes = set(targets.values())
node_to_comm = {n: i for i, comm in enumerate(communities) for n in comm}

# Reuse k-core values if the previous section has already computed them.
try:
    core_lookup = core_series.to_dict()
except NameError:
    U_simple = U.copy()
    U_simple.remove_edges_from(nx.selfloop_edges(U_simple))
    core_lookup = nx.core_number(U_simple)

target_metrics = []
for short, node in targets.items():
    row = {
        'target': short,
        'graph_node': node,
        'abbr': variable_meta.get(node, {}).get('abbr'),
        'full_name': variable_meta.get(node, {}).get('name'),
        'community': node_to_comm[node],
        'var_type': G.nodes[node].get('var_type'),
        'in_degree': G.in_degree(node),
        'out_degree': G.out_degree(node),
        'downstream_reach': len(nx.descendants(G, node)),
        'betweenness': centrality.loc[node, 'betweenness'],
        'pagerank': centrality.loc[node, 'pagerank'],
        'eigenvector': centrality.loc[node, 'eigenvector'],
        'core': core_lookup[node],
    }
    target_metrics.append(row)

target_metrics_df = pd.DataFrame(target_metrics).set_index('target')
print('Target nodes: community membership and own network metrics')
display(target_metrics_df.round(4))
target_metrics_df.to_csv('target_network_metrics.csv')


Target nodes: community membership and own network metrics


,graph_node,abbr,full_name,community,var_type,in_degree,out_degree,downstream_reach,betweenness,pagerank,eigenvector,core
target,,,,,,,,,,,,
nr,nonrenewable_resources,NR,Nonrenewable Resources,11,stateful,2,1,172,0.0161,0.0019,0.0091,2
pop,population,POP,population,1,component,4,10,172,0.1482,0.0082,0.0695,3
ppolx,persistent_pollution_index,PPOLX,persistent pollution index,9,component,2,4,172,0.0636,0.0067,0.0049,2
le,life_expectancy,LE,life expectancy,1,component,5,7,172,0.1276,0.0124,0.0283,3


In [51]:
# Correlation among the four target time series in the baseline World3 run.
# This is not used to choose policy levers directly; it shows whether the target outcomes carry overlapping information during the collapse-relevant period.
try:
    import pyworld3

    def run_world3_baseline(pyear=1975, year_max=2100, **constants):
        w3 = pyworld3.World3(year_max=year_max, pyear=pyear)
        w3.init_world3_constants(**constants)
        w3.init_world3_variables()
        w3.set_world3_table_functions()
        w3.set_world3_delay_functions()
        w3.run_world3()
        return w3

    baseline = run_world3_baseline()
    t = np.arange(baseline.year_min, baseline.year_max + baseline.dt, baseline.dt)
    mask = t >= 2000

    target_series = pd.DataFrame({
        'nr': baseline.nr[mask] / baseline.nr[0],
        'pop': baseline.pop[mask] / 1e9,
        'ppolx': baseline.ppolx[mask],
        'le': baseline.le[mask],
    }, index=t[mask])

    target_corr = target_series.corr().round(2)
    print('Baseline target correlation, 2000-2100')
    display(target_corr)
    target_corr.to_csv('target_correlation.csv')
except Exception as e:
    print('Could not run pyworld3 baseline for target correlations:')
    print(repr(e))
    target_corr = None


Baseline target correlation, 2000-2100


,nr,pop,ppolx,le
nr,1.00,0.46,0.22,0.83
pop,0.46,1.00,0.95,0.83
ppolx,0.22,0.95,1.00,0.65
le,0.83,0.83,0.65,1.00


### Upstream search logic

Because graph edges are encoded as `dependency -> variable`, a candidate that can affect a target must be a directed ancestor of that target. The table below lists upstream variables that can reach at least one of the four sustainability targets, with distance and centrality metrics attached.

Important columns:

- `target_count`: how many of the four targets can be reached through directed paths.
- `dist_*`: shortest directed path length from the upstream variable to each target.
- `avg_dist`: the average shortest-path distance across the reachable targets.
- `downstream_reach`: number of variables reachable from this node.
- `betweenness`, `pagerank`, `core`: complementary structural importance metrics.
- `community`: subsystem label from modularity detection.

To keep the candidate set focused on plausible influence paths, the upstream search only keeps variables whose shortest-path distance to at least one target is at most 10 steps.


In [52]:
# Directed ancestors of the four sustainability targets
MAX_TARGET_DISTANCE = 10
all_target_ancestors = set().union(*(nx.ancestors(G, node) for node in targets.values())) - target_nodes

ancestor_rows = []
for node in sorted(all_target_ancestors):
    dists = {}
    for short, target_node in targets.items():
        if node != target_node and nx.has_path(G, node, target_node):
            d = nx.shortest_path_length(G, node, target_node)
            if 0 < d <= MAX_TARGET_DISTANCE:
                dists[short] = d
    if not dists:
        continue

    ancestor_rows.append({
        'node': node,
        'abbr': variable_meta.get(node, {}).get('abbr'),
        'full_name': variable_meta.get(node, {}).get('name'),
        'var_type': G.nodes[node].get('var_type'),
        'community': node_to_comm[node],
        'target_count': len(dists),
        'targets_reached': ','.join(dists.keys()),
        'avg_dist': np.mean(list(dists.values())),
        'out_degree': G.out_degree(node),
        'in_degree': G.in_degree(node),
        'downstream_reach': len(nx.descendants(G, node)),
        'betweenness': centrality.loc[node, 'betweenness'],
        'pagerank': centrality.loc[node, 'pagerank'],
        'eigenvector': centrality.loc[node, 'eigenvector'],
        'core': core_lookup[node],
        **{f'dist_{short}': dists.get(short, np.nan) for short in targets},
    })

upstream_df = pd.DataFrame(ancestor_rows)
upstream_df = upstream_df.sort_values(
    ['target_count', 'avg_dist', 'downstream_reach', 'betweenness'],
    ascending=[False, True, False, False]
).reset_index(drop=True)

print(f'Upstream variables that can reach at least one target within {MAX_TARGET_DISTANCE} steps: {len(upstream_df)}')
print('Top upstream variables by target coverage, distance, reach, and betweenness:')
display(upstream_df.head(30).round(3))
upstream_df.to_csv('target_upstream_variables_all.csv', index=False)


Upstream variables that can reach at least one target within 10 steps: 270
Top upstream variables by target coverage, distance, reach, and betweenness:


,node,abbr,full_name,var_type,community,target_count,targets_reached,avg_dist,out_degree,in_degree,downstream_reach,betweenness,pagerank,eigenvector,core,dist_nr,dist_pop,dist_ppolx,dist_le
0,time,NaN,TIME,other,2,4,"nr,pop,ppolx,le",3.25,24,0,178,0.000,0.001,0.490,3,3.0,3.0,5.0,2.0
1,population_0_to_14,P1,Population 0 To 14,stateful,1,4,"nr,pop,ppolx,le",3.50,3,4,172,0.081,0.013,0.030,3,3.0,1.0,6.0,4.0
2,population_15_to_44,P2,Population 15 To 44,stateful,1,4,"nr,pop,ppolx,le",3.50,5,4,172,0.035,0.006,0.032,3,3.0,1.0,6.0,4.0
3,population_45_to_64,P3,Population 45 To 64,stateful,1,4,"nr,pop,ppolx,le",3.50,4,4,172,0.030,0.005,0.015,3,3.0,1.0,6.0,4.0
4,population_65_plus,P4,Population 65 Plus,stateful,1,4,"nr,pop,ppolx,le",3.50,2,3,172,0.020,0.005,0.013,3,3.0,1.0,6.0,4.0
5,gdp_pc_unit,NaN,GDP pc unit,constant,0,4,"nr,pop,ppolx,le",4.25,16,0,173,0.000,0.001,0.109,3,3.0,5.0,6.0,3.0
6,initial_population_0_to_14,P1I,initial population 0 to 14,constant,1,4,"nr,pop,ppolx,le",4.50,1,0,173,0.000,0.001,0.005,1,4.0,2.0,7.0,5.0
7,initial_population_15_to_44,P2I,initial population 15 to 44,constant,1,4,"nr,pop,ppolx,le",4.50,1,0,173,0.000,0.001,0.005,1,4.0,2.0,7.0,5.0
8,initial_population_54_to_64,P3I,initial population 54 to 64,constant,1,4,"nr,pop,ppolx,le",4.50,1,0,173,0.000,0.001,0.002,1,4.0,2.0,7.0,5.0
9,initial_population_65_plus,P4I,initial population 65 plus,constant,1,4,"nr,pop,ppolx,le",4.50,1,0,173,0.000,0.001,0.002,1,4.0,2.0,7.0,5.0


### Filtering upstream variables into policy candidates

The upstream table includes many variables that are structurally important but should not be grid-search levers, such as target outcomes, initial conditions, unit constants, and endogenous stocks/flows that are better interpreted as model outputs.

For candidate policy factors, this section applies a stricter filter:

1. The variable must be a directed upstream ancestor of at least two sustainability targets.
2. It must map to a grid-searchable World3/PyWorld3 parameter (`grid_param`) via the full-name/abbreviation metadata in `world3-03_variables.json`.
3. Initial conditions, unit/bookkeeping constants, and the four target variables themselves are excluded.
4. Preference is given to interpretable policy controls: policy timing, post-policy factors, delays, technology/productivity factors, allocation fractions, and desired social/behavioral parameters.
5. After scoring, only the top 15 candidate variables are retained for the final table and later grid-search design.

The leverage score used below is the unweighted additive score:

`leverage_score = coverage_n + distance_n + betweenness_n + feedbackimportance_n + core_n`

where:

- `coverage_n = coverage / 4`: normalized target coverage, where `coverage` is the actual number of sustainability targets reached by directed paths.
- `distance_n = 1 - minmax(avg_dist)`: rewards candidates that are closer to the targets on average.
- `betweenness_n = minmax(betweenness)`: rewards candidates that sit on more shortest paths and can bridge subsystems.
- `feedbackimportance_n = minmax(feedbackimportance)`, with `feedbackimportance = cycle_count / total_cycle_count`: rewards candidates that participate in more directed feedback cycles relative to the whole World3 network.
- `core_n = minmax(core)`: rewards candidates embedded in the denser structural backbone.

Each term now enters with coefficient 1, so the ranking treats target coverage, distance, bridging, feedback participation, and core embeddedness as equally important structural signals.


In [53]:
# Build a map from graph nodes to grid-search parameter names.
fallback_world3_params = {
    'pyear',
    'p1i', 'p2i', 'p3i', 'p4i', 'dcfsn', 'fcest', 'hsid', 'ieat', 'len',
    'lpd', 'mtfn', 'pet', 'rlt', 'sad', 'zpgt',
    'ici', 'sci', 'iet', 'iopcd', 'lfpf', 'lufdt', 'icor1', 'icor2', 'scor1',
    'scor2', 'alic1', 'alic2', 'alsc1', 'alsc2', 'fioac1', 'fioac2',
    'ali', 'pali', 'lfh', 'palt', 'pl', 'alai1', 'alai2', 'io70', 'lyf1',
    'lyf2', 'sd', 'uili', 'alln', 'uildt', 'lferti', 'ilf', 'fspd', 'sfpc',
    'ppoli', 'ppol70', 'ahl70', 'amti', 'imti', 'imef', 'fipm', 'frpm',
    'ppgf1', 'ppgf2', 'ppgf21', 'pptd1', 'pptd2', 'nri', 'nruf1', 'nruf2'
}
try:
    import pyworld3
    world3_param_names = set(inspect.signature(pyworld3.World3.init_world3_constants).parameters)
    world3_param_names.discard('self')
    world3_param_names.add('pyear')
except Exception:
    world3_param_names = fallback_world3_params

def infer_grid_param(node):
    if node == 'policy_year':
        return 'pyear'
    abbr = variable_meta.get(node, {}).get('abbr')
    if isinstance(abbr, str):
        candidate = abbr.lower()
        if candidate in world3_param_names:
            return candidate
    return np.nan

upstream_df['grid_param'] = upstream_df['node'].map(infer_grid_param)

exclude_prefixes = ('initial_', 'unit_')
exclude_exact = {
    'time', 'one_year', 'zero', 'thousand', 'gdp_pc_unit', 'ha_per_gha',
    'nonrenewable_resources', 'population', 'persistent_pollution_index', 'life_expectancy',
    'life_expectancy_normal', 'subsistence_food_per_capita',
    'persistent_pollution_in_1970', 'initial_nonrenewable_resources',
    'initial_persistent_pollution', 'population_equilibrium_time',
    'reproductive_lifetime', 'maximum_total_fertility_normal',
}
policy_keywords = (
    'policy', 'factor', 'ratio', 'delay', 'desired', 'technology', 'allocation',
    'fraction', 'life', 'yield', 'fertility', 'capital', 'pollution', 'resource',
    'services', 'consumption',
)

def is_policy_candidate(row):
    node = row['node']
    if pd.isna(row['grid_param']):
        return False
    if node in exclude_exact or any(node.startswith(p) for p in exclude_prefixes):
        return False
    if row['target_count'] < 2:
        return False
    text = node.lower()
    return any(k in text for k in policy_keywords)

policy_candidates = upstream_df[upstream_df.apply(is_policy_candidate, axis=1)].copy()

def minmax(series):
    series = series.astype(float)
    if series.max() == series.min():
        return pd.Series(1.0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

policy_candidates['coverage'] = policy_candidates['target_count'].astype(float)
policy_candidates['coverage_n'] = policy_candidates['coverage'] / len(targets)
policy_candidates['distance_n'] = 1 - minmax(policy_candidates['avg_dist'])
policy_candidates['betweenness_n'] = minmax(policy_candidates['betweenness'])
policy_candidates['core_n'] = minmax(policy_candidates['core'])

candidate_cycle_counts = {node: 0 for node in policy_candidates['node']}
total_cycle_count = len(cycles)
for cycle in cycles:
    for node in cycle:
        if node in candidate_cycle_counts:
            candidate_cycle_counts[node] += 1

policy_candidates['cycle_count'] = policy_candidates['node'].map(candidate_cycle_counts).fillna(0).astype(int)
if total_cycle_count == 0:
    policy_candidates['feedbackimportance'] = 0.0
else:
    policy_candidates['feedbackimportance'] = policy_candidates['cycle_count'] / total_cycle_count
policy_candidates['feedbackimportance_n'] = minmax(policy_candidates['feedbackimportance'])

policy_candidates['leverage_score'] = (
    policy_candidates['coverage_n'] +
    policy_candidates['distance_n'] +
    policy_candidates['betweenness_n'] +
    policy_candidates['feedbackimportance_n'] +
    policy_candidates['core_n']
)

policy_candidates = policy_candidates.sort_values(
    ['leverage_score', 'coverage', 'avg_dist', 'feedbackimportance', 'betweenness'],
    ascending=[False, False, True, False, False]
).head(15).reset_index(drop=True)

cols = [
    'grid_param', 'node', 'abbr', 'full_name', 'var_type', 'community',
    'target_count', 'coverage', 'coverage_n', 'targets_reached', 'dist_nr', 'dist_pop', 'dist_ppolx', 'dist_le',
    'avg_dist', 'out_degree', 'downstream_reach', 'betweenness', 'pagerank',
    'eigenvector', 'core', 'cycle_count', 'feedbackimportance', 'feedbackimportance_n', 'distance_n',
    'betweenness_n', 'core_n', 'leverage_score'
]
print(f'Policy-candidate upstream variables retained after leverage-score filtering: {len(policy_candidates)}')
print(f'Total directed simple cycles in the World3 graph: {total_cycle_count}')
display(policy_candidates[cols].round(6))
policy_candidates[cols].to_csv('target_policy_candidate_scores_top15.csv', index=False)


Policy-candidate upstream variables retained after leverage-score filtering: 15
Total directed simple cycles in the World3 graph: 1863803


,grid_param,node,abbr,full_name,var_type,community,target_count,coverage,coverage_n,targets_reached,...,pagerank,eigenvector,core,cycle_count,feedbackimportance,feedbackimportance_n,distance_n,betweenness_n,core_n,leverage_score
0,icor2,industrial_capital_output_ratio_2,ICOR2,industrial capital output ratio 2,component,3,4,4.0,1.00,"nr,pop,ppolx,le",...,0.007359,0.037755,2,581059,0.311760,1.000000,0.562500,1.000000,0.5,4.062500
1,pyear,policy_year,PYEAR,POLICY YEAR,constant,2,4,4.0,1.00,"nr,pop,ppolx,le",...,0.000649,0.368726,3,0,0.000000,0.000000,1.000000,0.000000,1.0,3.000000
2,ppgf2,persistent_pollution_generation_factor_2,PPGF2,persistent pollution generation factor 2,stateful,3,3,3.0,0.75,"nr,ppolx,le",...,0.003188,0.029411,2,336022,0.180288,0.578292,0.750000,0.380322,0.5,2.958615
3,nruf2,resource_use_fact_2,NRUF2,resource use fact 2,stateful,3,2,2.0,0.50,"nr,le",...,0.002997,0.028699,2,118520,0.063590,0.203972,1.000000,0.401548,0.5,2.605520
4,lyf2,land_yield_factor_2,LYF2,land yield factor 2,stateful,3,3,3.0,0.75,"nr,pop,le",...,0.002809,0.028979,2,235932,0.126586,0.406038,0.416667,0.394428,0.5,2.467133
5,hsid,health_services_impact_delay,HSID,health services impact delay,constant,0,3,3.0,0.75,"nr,pop,le",...,0.000649,0.004991,2,0,0.000000,0.000000,1.000000,0.000000,0.5,2.250000
6,fcest,fertility_control_effectiveness_time,FCEST,fertility control effectiveness time,constant,0,4,4.0,1.00,"nr,pop,ppolx,le",...,0.000649,0.015774,1,0,0.000000,0.000000,0.625000,0.000000,0.0,1.625000
7,ahl70,assimilation_half_life_in_1970,AHL70,assimilation half life in 1970,constant,9,3,3.0,0.75,"pop,ppolx,le",...,0.000649,0.000025,1,0,0.000000,0.000000,0.833333,0.000000,0.0,1.583333
8,icor1,industrial_capital_output_ratio_1,ICOR1,industrial capital output ratio 1,constant,2,4,4.0,1.00,"nr,pop,ppolx,le",...,0.000649,0.024279,1,0,0.000000,0.000000,0.562500,0.000000,0.0,1.562500
9,ppgf1,persistent_pollution_generation_factor_1,PPGF1,persistent pollution generation factor 1,constant,3,3,3.0,0.75,"nr,ppolx,le",...,0.000649,0.023551,1,0,0.000000,0.000000,0.750000,0.000000,0.0,1.500000


In [54]:
# Final top-4 influence factors from the top-15 leverage-score candidate set.
final_top4 = policy_candidates.head(4).copy()

print('Recommended 4 influence factors from the leverage-score ranking:')
display(final_top4[cols].round(6))
print('Parameter list:', final_top4['grid_param'].tolist())
final_top4[cols].to_csv('recommended_grid_search_factors_top4.csv', index=False)


Recommended 4 influence factors from the leverage-score ranking:


,grid_param,node,abbr,full_name,var_type,community,target_count,coverage,coverage_n,targets_reached,...,pagerank,eigenvector,core,cycle_count,feedbackimportance,feedbackimportance_n,distance_n,betweenness_n,core_n,leverage_score
0,icor2,industrial_capital_output_ratio_2,ICOR2,industrial capital output ratio 2,component,3,4,4.0,1.00,"nr,pop,ppolx,le",...,0.007359,0.037755,2,581059,0.311760,1.000000,0.5625,1.000000,0.5,4.062500
1,pyear,policy_year,PYEAR,POLICY YEAR,constant,2,4,4.0,1.00,"nr,pop,ppolx,le",...,0.000649,0.368726,3,0,0.000000,0.000000,1.0000,0.000000,1.0,3.000000
2,ppgf2,persistent_pollution_generation_factor_2,PPGF2,persistent pollution generation factor 2,stateful,3,3,3.0,0.75,"nr,ppolx,le",...,0.003188,0.029411,2,336022,0.180288,0.578292,0.7500,0.380322,0.5,2.958615
3,nruf2,resource_use_fact_2,NRUF2,resource use fact 2,stateful,3,2,2.0,0.50,"nr,le",...,0.002997,0.028699,2,118520,0.063590,0.203972,1.0000,0.401548,0.5,2.605520


Parameter list: ['icor2', 'pyear', 'ppgf2', 'nruf2']


### Interpretation of the final candidate set

The notebook now uses a leverage-score screening rule. 

Candidate variables are first filtered to plausible policy parameters, then ranked by equal-weight structural signals: target coverage, short directed distance to the targets, bridge importance, participation in feedback cycles, and `k`-core embeddedness.

Only the top 15 candidates are retained in the final screening table. The top 4 of those 15 form the recommended parameter set for the next grid-search stage.
